# Phase 4: Biased SVD Training

This notebook trains the Biased SVD model with sentiment bias terms.

**Equation**:
$$r_{ui} = \overline{r} + w_1(\mu_u - \overline{r}) + w_2(\mu_i - \overline{r}) + w_3(\mu_{u,s} - \overline{r}) + w_4(\mu_{i,s} - \overline{r}) + b_u + b_i + \langle p_u, q_i \rangle$$

**Run Time**: ~20 mins on GPU
**Input**: Outputs from Phase 2-3
**Output**: Trained SVD model + matrices

In [1]:
# Cell 1: Install dependencies
!pip install -q torch pandas numpy scikit-learn tqdm -q

In [2]:
# Cell 2: Imports & Setup
import os
import pickle
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

# Environment detection - local vs Kaggle
BASE_DIR = os.getcwd()
if os.path.exists('/kaggle/input/datasets/chandrimanandi/phase-2-3-results/train.csv'):
    # Real Kaggle environment
    INPUT_DIR = '/kaggle/input/datasets/chandrimanandi/phase-2-3-results'
    OUTPUT_DIR = '/kaggle/working'
else:
    # Local environment
    INPUT_DIR = os.path.join(BASE_DIR, 'output')
    OUTPUT_DIR = os.path.join(BASE_DIR, 'output', 'phase-4-results')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"INPUT_DIR: {INPUT_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

Device: cuda
INPUT_DIR: /kaggle/input/datasets/chandrimanandi/phase-2-3-results
OUTPUT_DIR: /kaggle/working


In [3]:
# Cell 3: Load preprocessed data
print("Loading Phase 2-3 outputs...")

train_df = pd.read_csv(f"{INPUT_DIR}/train.csv")
val_df = pd.read_csv(f"{INPUT_DIR}/val.csv")
test_df = pd.read_csv(f"{INPUT_DIR}/test.csv")

with open(f"{INPUT_DIR}/id_maps.pkl", 'rb') as f:
    id_maps = pickle.load(f)

with open(f"{INPUT_DIR}/rating_stats.pkl", 'rb') as f:
    rating_stats = pickle.load(f)

with open(f"{INPUT_DIR}/sentiment_stats.pkl", 'rb') as f:
    sentiment_stats = pickle.load(f)

n_users = id_maps['n_users']
n_items = id_maps['n_items']
global_mean = rating_stats['global_mean']

print(f"Users: {n_users}, Items: {n_items}")
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Loading Phase 2-3 outputs...
Users: 5541, Items: 3568
Train: 53624, Val: 5541, Test: 5541


In [4]:
# Cell 4: PyTorch SVD model
class BiasedSVD(nn.Module):
    def __init__(self, n_users, n_items, k_factors, global_mean):
        super().__init__()
        self.n_users = n_users
        self.n_items = n_items
        self.k = k_factors
        self.global_mean = global_mean
        
        # User/item factors
        self.P = nn.Parameter(torch.randn(n_users, k_factors) * 0.01)
        self.Q = nn.Parameter(torch.randn(n_items, k_factors) * 0.01)
        
        # Biases
        self.bu = nn.Parameter(torch.zeros(n_users))
        self.bi = nn.Parameter(torch.zeros(n_items))
        
        # Sentiment bias weights
        self.w_user_rating = nn.Parameter(torch.ones(1) * 0.5)
        self.w_item_rating = nn.Parameter(torch.ones(1) * 0.5)
        self.w_user_sentiment = nn.Parameter(torch.ones(1) * 0.2)
        self.w_item_sentiment = nn.Parameter(torch.ones(1) * 0.2)
    
    def forward(self, user_idx, item_idx, u_rating_mean, i_rating_mean, u_sent_mean, i_sent_mean):
        """
        Args:
            user_idx: (batch,)
            item_idx: (batch,)
            u_rating_mean: (batch,)
            i_rating_mean: (batch,)
            u_sent_mean: (batch,)
            i_sent_mean: (batch,)
        """
        # SVD term
        p_u = self.P[user_idx]  # (batch, k)
        q_i = self.Q[item_idx]  # (batch, k)
        svd_term = (p_u * q_i).sum(dim=1)  # (batch,)
        
        # Biases
        bias_term = self.bu[user_idx] + self.bi[item_idx]
        
        # Sentiment terms
        sent_term = (
            self.w_user_rating * (u_rating_mean - self.global_mean) +
            self.w_item_rating * (i_rating_mean - self.global_mean) +
            self.w_user_sentiment * u_sent_mean +
            self.w_item_sentiment * i_sent_mean
        )
        
        prediction = self.global_mean + svd_term + bias_term + sent_term
        return prediction

K_FACTORS = 20
model = BiasedSVD(n_users, n_items, K_FACTORS, global_mean).to(device)
print(f"Model created with {K_FACTORS} latent factors")

Model created with 20 latent factors


In [5]:
# Cell 5: Training setup
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.001)

MAX_EPOCHS = 50
BATCH_SIZE = 256
PATIENCE = 10

print(f"Learning rate: 0.001")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max epochs: {MAX_EPOCHS}")
print(f"Early stopping patience: {PATIENCE}")

Learning rate: 0.001
Batch size: 256
Max epochs: 50
Early stopping patience: 10


In [6]:
# Cell 6: Data preparation
def prepare_batch(df, batch_size, device):
    """Convert DataFrame to PyTorch tensors."""
    user_idx = torch.LongTensor(df['user_idx'].values).to(device)
    item_idx = torch.LongTensor(df['item_idx'].values).to(device)
    ratings = torch.FloatTensor(df['rating'].values).to(device)
    u_rate_mean = torch.FloatTensor(df['user_rating_mean'].values).to(device)
    i_rate_mean = torch.FloatTensor(df['item_rating_mean'].values).to(device)
    u_sent_mean = torch.FloatTensor(df['user_sentiment_mean'].values).to(device)
    i_sent_mean = torch.FloatTensor(df['item_sentiment_mean'].values).to(device)
    
    n_batches = (len(df) + batch_size - 1) // batch_size
    batches = []
    
    for i in range(n_batches):
        start = i * batch_size
        end = min((i+1) * batch_size, len(df))
        batches.append((
            user_idx[start:end],
            item_idx[start:end],
            ratings[start:end],
            u_rate_mean[start:end],
            i_rate_mean[start:end],
            u_sent_mean[start:end],
            i_sent_mean[start:end],
        ))
    return batches

train_batches = prepare_batch(train_df, BATCH_SIZE, device)
val_batches = prepare_batch(val_df, BATCH_SIZE, device)
print(f"Train batches: {len(train_batches)}")
print(f"Val batches: {len(val_batches)}")

Train batches: 210
Val batches: 22


In [7]:
# Cell 7: Training loop
print("\n" + "="*60)
print("TRAINING BIASED SVD")
print("="*60)

best_val_rmse = float('inf')
patience_counter = 0
train_losses = []
val_rmses = []

for epoch in range(MAX_EPOCHS):
    # Training
    model.train()
    train_loss = 0.0
    
    for batch in train_batches:
        u_idx, i_idx, ratings, u_rate, i_rate, u_sent, i_sent = batch
        
        optimizer.zero_grad()
        predictions = model(u_idx, i_idx, u_rate, i_rate, u_sent, i_sent)
        loss = criterion(predictions, ratings)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * len(ratings)
    
    train_loss /= len(train_df)
    train_rmse = np.sqrt(train_loss)
    train_losses.append(train_rmse)
    
    # Validation
    model.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for batch in val_batches:
            u_idx, i_idx, ratings, u_rate, i_rate, u_sent, i_sent = batch
            predictions = model(u_idx, i_idx, u_rate, i_rate, u_sent, i_sent)
            loss = criterion(predictions, ratings)
            val_loss += loss.item() * len(ratings)
    
    val_loss /= len(val_df)
    val_rmse = np.sqrt(val_loss)
    val_rmses.append(val_rmse)
    
    # Early stopping
    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        patience_counter = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d} | Train RMSE: {train_rmse:.4f} | Val RMSE: {val_rmse:.4f}")
    
    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        model.load_state_dict(best_state)
        break

print(f"\nBest val RMSE: {best_val_rmse:.4f}")


TRAINING BIASED SVD
Epoch   1 | Train RMSE: 0.8340 | Val RMSE: 0.9498
Epoch   5 | Train RMSE: 0.7979 | Val RMSE: 0.9611
Epoch  10 | Train RMSE: 0.7939 | Val RMSE: 0.9619
Early stopping at epoch 11

Best val RMSE: 0.9498


In [8]:
# Cell 8: Test evaluation
print("\n" + "="*60)
print("TEST EVALUATION")
print("="*60)

model.eval()
test_loss = 0.0
test_mae = 0.0

with torch.no_grad():
    for batch in val_batches:  # Use val_batches preparation but test_df logic
        pass

# Proper test evaluation
test_batches = prepare_batch(test_df, BATCH_SIZE, device)

with torch.no_grad():
    for batch in test_batches:
        u_idx, i_idx, ratings, u_rate, i_rate, u_sent, i_sent = batch
        predictions = model(u_idx, i_idx, u_rate, i_rate, u_sent, i_sent)
        
        # Clamp predictions to rating range
        predictions = torch.clamp(predictions, 1, 5)
        
        test_loss += criterion(predictions, ratings).item() * len(ratings)
        test_mae += (predictions - ratings).abs().sum().item()

test_rmse = np.sqrt(test_loss / len(test_df))
test_mae = test_mae / len(test_df)

print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test MAE:  {test_mae:.4f}")


TEST EVALUATION
Test RMSE: 0.9863
Test MAE:  0.7068


In [9]:
# Cell 9: Extract model components
print("\nExtracting model components...")

P = model.P.detach().cpu().numpy()  # (n_users, k)
Q = model.Q.detach().cpu().numpy()  # (n_items, k)
bu = model.bu.detach().cpu().numpy()  # (n_users,)
bi = model.bi.detach().cpu().numpy()  # (n_items,)

print(f"P shape: {P.shape}")
print(f"Q shape: {Q.shape}")
print(f"bu shape: {bu.shape}")
print(f"bi shape: {bi.shape}")


Extracting model components...
P shape: (5541, 20)
Q shape: (3568, 20)
bu shape: (5541,)
bi shape: (3568,)


In [ ]:
# Cell 9: Save all outputs
print("\n" + "="*60)
print("SAVING OUTPUTS")
print("="*60)

# Extract model components
P = model.P.detach().cpu().numpy()  # (n_users, k)
Q = model.Q.detach().cpu().numpy()  # (n_items, k)
bu = model.bu.detach().cpu().numpy()  # (n_users,)
bi = model.bi.detach().cpu().numpy()  # (n_items,)

# Save complete model
svd_model = {
    'P': P,
    'Q': Q,
    'bu': bu,
    'bi': bi,
    'global_mean': float(global_mean),
    'k_factors': K_FACTORS,
    'n_users': n_users,
    'n_items': n_items,
}

with open(f"{OUTPUT_DIR}/biased_svd_model.pkl", 'wb') as f:
    pickle.dump(svd_model, f)
print("✓ Biased SVD model saved")

# Save training history
with open(f"{OUTPUT_DIR}/training_history.pkl", 'wb') as f:
    pickle.dump({
        'train_rmse': train_losses,
        'val_rmse': val_rmses,
    }, f)
print("✓ Training history saved")

# Save metrics
import json
metrics = {
    'val_rmse': float(best_val_rmse),
    'test_rmse': float(test_rmse),
    'test_mae': float(test_mae),
}

with open(f"{OUTPUT_DIR}/svd_metrics.json", 'w') as f:
    json.dump(metrics, f, indent=2)
print("✓ Metrics saved")

print(f"\n{'='*60}")
print(f"PHASE 4 COMPLETE")
print(f"{'='*60}")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test MAE:  {test_mae:.4f}")
print(f"\nAll files saved to: {OUTPUT_DIR}")



SAVING OUTPUTS
✓ Biased SVD model saved
✓ Training history saved
✓ Metrics saved

PHASE 4 COMPLETE
Test RMSE: 0.9863
Test MAE:  0.7068

All files saved to: /kaggle/working
